# GeoMamba — (d) TAM GEOMETRİK: planlanan ama kurulmayan basamak (Kaggle T4)

Eşdeğişkenlik **eğitimden önce sayısal olarak doğrulandı**
(`tests/test_equivariance.py`, `model(x Mᵀ) == model(x) Mᵀ`):

| model | göreli hata |
|---|---|
| **yeni `full_geo` (d)** | **3e-16 … 1e-15** |
| eski "fullgeo" (`geo_proj`) | 0.93 |
| rotor+slot (c) | 0.28 |
| vanilla mamba | 1.08 |

Eski "fullgeo" kolu eşdeğişkenlik bakımından vanilla Mamba'dan ayırt
edilemiyordu; ona dayanan "Clifford karışımı zararlı" hükmü geri çekildi.

**Protokolde iki düzeltme:**
1. **Gerçekten az veri.** N=300 × 257 adım ≈ 77.000 geçiş — bu az veri değil,
   bir inductive bias'ın satın alacağı bir şey yok. N=10 ≈ 2.500 geçiş.
   Bias bir yerde işe yarayacaksa burada yarayacak.
2. Her koşu döndürülmüş test setinde de skorlanıyor (`--rot-test`).
   `full_geo` inşaat gereği `rot_gap ≈ 1.0` vermeli, diğerleri vermemeli —
   eğitilmiş model üzerinde çalışan yerleşik bir geçerlilik kontrolü.

**Kollar** (parametre eşit, +%5.5 içinde): `mamba` 128.2k · `mambaslot` 125.1k ·
`rotorslot` 128.9k · `fullgeo` 135.3k · `fullgeonb` (geometrik çarpım yok) 128.5k

N ∈ {10, 30, 100, 300} × 5 seed × 5 kol = **100 koşu**, T4×2'de ~3 saat.

**Kurulum:** Session options → GPU **T4 x2**, Internet **On**; Add-ons →
Secrets → `GH_TOKEN`. Sonra **Save Version → Save & Run All (Commit)**.
İnteraktif Run All ile çalıştırmayın, çıktı kaybolur.


In [ ]:
%cd /kaggle/working
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
!rm -rf /kaggle/working/geomamba
!git clone --depth 1 https://{token}@github.com/yusuftiryaki/geomamba.git /kaggle/working/geomamba


In [ ]:
%cd /kaggle/working/geomamba
import os, torch
os.environ["PYTHONPATH"] = "/kaggle/working/geomamba/src"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Session options'tan GPU secin!")


In [ ]:
import os, subprocess, sys, pathlib
GRID = "configs/grid/m6_fullgeo.yaml"

n_gpu = int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU "))
print("GPU sayisi:", n_gpu)

# datasets once, then one process per GPU
subprocess.run([sys.executable, "scripts/run_grid.py", "--grid", GRID, "--data-only"], check=True)
procs = []
for k in range(max(n_gpu, 1)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    log = open(f"shard{k}.log", "w")
    procs.append((k, log, subprocess.Popen(
        [sys.executable, "scripts/run_grid.py", "--grid", GRID, "--skip-data",
         "--shard", f"{k}/{max(n_gpu, 1)}"],
        env=env, stdout=log, stderr=subprocess.STDOUT)))
print(f"{len(procs)} shard baslatildi; loglar shard*.log")
for k, log, p in procs:
    rc = p.wait(); log.close()
    print(f"shard {k} bitti, rc={rc}")
    print(pathlib.Path(f"shard{k}.log").read_text()[-2000:])


In [ ]:
import glob, shutil, pandas as pd
# Shard sonuçları iki yere yazılır: koşu sırasında artımlı olarak
# /kaggle/working/grid_results-shard*.csv, shard TAMAMEN bitince de
# experiments/ altına. Shard zaman aşımına uğrarsa ikincisi hiç yazılmaz --
# o yüzden ikisine de bakıyoruz, yoksa tamamlanmış koşular kayıp görünür.
paths = sorted(set(glob.glob("experiments/grid_results*.csv")
                   + glob.glob("/kaggle/working/grid_results-shard*.csv")))
frames = []
for f in paths:
    try:
        frames.append(pd.read_csv(f))
    except Exception as e:
        print("okunamadi", f, e)
if frames:
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset="name", keep="last")
    df.to_csv("/kaggle/working/grid_results.csv", index=False)
    for f in glob.glob("experiments/grid_results*.json"):
        shutil.copy(f, "/kaggle/working/")
    print(f"{len(df)} kosu toplandi ({len(paths)} dosyadan)")
    display(df)
else:
    print("sonuc yok - shard*.log dosyalarina bakin")
    for f in sorted(glob.glob("shard*.log")):
        print("=" * 20, f)
        print(open(f).read()[-3000:])


In [ ]:
import os, pandas as pd
if not os.path.exists("/kaggle/working/grid_results.csv"):
    raise SystemExit("Sonuc dosyasi yok - onceki hucrenin ciktisina bakin; "
                     "shard'lar cokmus ya da hic kosu bitmemis olabilir.")
df = pd.read_csv("/kaggle/working/grid_results.csv")
df["arm"] = df.name.str.split("-").str[1]
df["n"]   = df.name.str.split("-").str[2].str[1:].astype(int)

print("=== rollout h100 (ortalama ± std, 5 seed) ===")
display(df.pivot_table(index="n", columns="arm", values="rollout_mse_h100",
                       aggfunc=["mean","std"]).round(4))
print("=== enerji kayması ===")
display(df.pivot_table(index="n", columns="arm", values="energy_drift", aggfunc="mean").round(3))
print("=== rot_gap: eşdeğişkenlik (ideal 1.0) — full_geo ~1.0 olmalı ===")
display(df.pivot_table(index="n", columns="arm", values="rot_gap", aggfunc="mean").round(3))
print("normalizasyon tabanı (persistence):", round(df.persistence_rot_gap.mean(), 3))

# eşleştirilmiş seed düelloları: geometrik çarpım ve eşdeğişkenlik ne kazandırıyor?
def duel(a, b, n, metric="rollout_mse_h100"):
    x = df[(df.arm==a)&(df.n==n)].sort_values("name")[metric].values
    y = df[(df.arm==b)&(df.n==n)].sort_values("name")[metric].values
    k = min(len(x), len(y))
    return f"{a} < {b}: {int((x[:k]<y[:k]).sum())}/{k}  ({x[:k].mean():.3f} vs {y[:k].mean():.3f})"

for n in sorted(df.n.unique()):
    print(f"\n--- N={n} ---")
    for a,b in [("fullgeo","mamba"),("fullgeo","rotorslot"),("fullgeo","mambaslot"),
                ("fullgeo","fullgeonb"),("fullgeonb","mamba")]:
        print(" ", duel(a,b,n))
